In [22]:
import pandas as pd

CUTOFF = pd.Timestamp("2026-01-16")

df = pd.read_parquet('../data/demand_enriched_corrupted.parquet')
print(df.shape)

(6330245, 32)


In [4]:
baseline = df[df['time_bucket'] < CUTOFF]   # clean historical window
corrupted = df[df['time_bucket'] >= CUTOFF]  # new potentially corrupted window

print(f"Baseline: {len(baseline)} rows")
print(f"Corrupted: {len(corrupted)} rows")

Baseline: 6079392 rows
Corrupted: 250853 rows


In [6]:
# Issue #1: Missing values: Increase in nulls in any column?
print(f"\nBaseline null rates:\n{baseline.isna().mean()}")
print(f"\nCorrupted null rates:\n{corrupted.isna().mean()}")

# Compute the differences in null rates between the baseline data and the corrupted data.
null_rate_differences = corrupted.isna().mean().values - baseline.isna().mean().values
print(f"\nNull rate differences:\n{null_rate_differences}")

# Count the number of positive differences (indicating increases in nulls).
num_pos_differences = null_rate_differences[null_rate_differences > 0].size
print(f"\nNumber of positive null rate differences:\n{num_pos_differences}")

# No issues here. ------------------------------------------


Baseline null rates:
PULocationID          0.000000
time_bucket           0.000000
trip_count            0.000000
hour                  0.000000
minute                0.000000
dayofweek             0.000000
is_weekend            0.000000
month                 0.000000
dayofyear             0.000000
weekofyear            0.000000
year                  0.000000
slot_of_day           0.000000
hour_sin              0.000000
hour_cos              0.000000
dow_sin               0.000000
dow_cos               0.000000
month_sin             0.000000
month_cos             0.000000
is_holiday            0.000000
cbd_pricing_active    0.000000
borough_id            0.000000
service_zone_id       0.000000
is_airport_zone       0.000000
zone_slot_baseline    0.000000
lag_15min             0.000009
lag_1h                0.000038
lag_2h                0.000075
lag_1day              0.000900
lag_1week             0.006301
roll_mean_1h          0.000009
roll_mean_2h          0.000009
roll_mean_1day   

In [9]:
# Issue #2: Outliers: Values outside expected ranges? (e.g., trip_count > 10x normal)

# Compute the standard deviation for each of the features of the baseline data.
baseline_std = baseline.std()

# Compute the mean for each of the features of the baseline data.
baseline_mean = baseline.mean()

# Compute the threshold value over which would be considered an outlier value.
outlier_threshold = baseline_mean + 3*baseline_std
negative_outlier_threshold = baseline_mean - 3*baseline_std

# Assess whether there are values in the corrupted data that exceed the outlier threshold value (or fall under the negative of the outlier threshold value).
corrupted_outlier_values_max = corrupted.max() - outlier_threshold
corrupted_outlier_values_min = corrupted.min() - negative_outlier_threshold

print(f"\nCorrupted max values difference with outlier threshold:\n{corrupted_outlier_values_max}")
print(f"\nCorrupted min values difference with outlier threshold:\n{corrupted_outlier_values_min}")

# Issue: Values are outside of the expected ranges (especially for trip_count)


Corrupted max values difference with outlier threshold:
PULocationID                             -98.349091
time_bucket           -363 days +08:10:18.247191896
trip_count                             99917.249193
hour                                      -9.266561
minute                                   -27.811534
dayofweek                                  -2.99932
is_weekend                                -0.640041
month                                    -14.900534
dayofyear                               -441.849536
weekofyear                               -62.789236
year                                      -0.555457
slot_of_day                              -35.633935
hour_sin                                  -1.121321
hour_cos                                  -1.121321
dow_sin                                   -1.147392
dow_cos                                   -1.122594
month_sin                                 -1.246504
month_cos                                 -1.275847
is_holi

In [15]:
# Issue #3: Duplicates: Rows repeated multiple times?
print(f"\nBaseline count of duplicates:\n{baseline.duplicated().sum()}")
print(f"\nCorrupted count of duplicates:\n{corrupted.duplicated().sum()}")

# Issue: Rows repeated multiple times in the corrupted data. ------------------------------------------


Baseline count of duplicates:
0

Corrupted count of duplicates:
8134


In [19]:
# Issue #4: Distribution shifts: Mean/std significantly different?
"""
print(f"\nBaseline means:\n{baseline.mean()}")
print(f"\nCorrupted means:\n{corrupted.mean()}")

print(f"\nBaseline stds:\n{baseline.std()}")
print(f"\nCorrupted stds:\n{corrupted.std()}")
"""

# Print the percentage differences between the means and standard deviations.
corrupted_means = corrupted.drop(columns=["time_bucket"]).mean()
baseline_means = baseline.drop(columns=["time_bucket"]).mean()
mean_percentage_diff = 100*((corrupted_means - baseline_means)/baseline_means)
print(f"\nMean percentage differences:\n{mean_percentage_diff}")

corrupted_stds = corrupted.drop(columns=["time_bucket"]).std()
baseline_stds = baseline.drop(columns=["time_bucket"]).std()
std_percentage_diff = 100*((corrupted_stds - baseline_stds)/baseline_stds)
print(f"\Std percentage differences:\n{std_percentage_diff}")


# Issue: Mean/std is significantly different in corrupted data. ------------------------------------------


Mean percentage differences:
PULocationID         -1.425021e+00
trip_count            3.618587e+02
hour                  1.993199e-03
minute                1.993199e-03
dayofweek             2.274742e+00
is_weekend            3.416645e+00
month                -7.439283e+01
dayofyear            -7.900349e+01
weekofyear           -7.599482e+01
year                  9.747880e-02
slot_of_day           1.993199e-03
hour_sin              5.574794e+13
hour_cos             -3.468299e+13
dow_sin              -2.525020e+03
dow_cos              -2.523926e+03
month_sin             3.305754e+04
month_cos             6.302954e+03
is_holiday            2.994652e+02
cbd_pricing_active    1.954787e+02
borough_id           -4.020283e+00
service_zone_id      -4.020283e+00
is_airport_zone      -4.020283e+00
zone_slot_baseline   -9.878060e-01
lag_15min            -2.025758e+01
lag_1h               -2.029810e+01
lag_2h               -2.030860e+01
lag_1day             -2.002936e+01
lag_1week            -1.6

In [17]:
# Issue #5: Schema changes: New/missing columns?
print(f"\nBaseline number of columns:\n{len(baseline.columns.tolist())}")
print(f"\nCorrupted number of columns:\n{len(corrupted.columns.tolist())}")

# Verify that the column names are the same (just because the counts are the same does not mean that all of the column names are necessarily the same).
print(f"\nBaseline columns: {baseline.columns.tolist()}")
print(f"\nCorrupted columns: {corrupted.columns.tolist()}")

# No issues here. ------------------------------------------


Baseline number of columns:
32

Corrupted number of columns:
32

Baseline columns: ['PULocationID', 'time_bucket', 'trip_count', 'hour', 'minute', 'dayofweek', 'is_weekend', 'month', 'dayofyear', 'weekofyear', 'year', 'slot_of_day', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_holiday', 'cbd_pricing_active', 'borough_id', 'service_zone_id', 'is_airport_zone', 'zone_slot_baseline', 'lag_15min', 'lag_1h', 'lag_2h', 'lag_1day', 'lag_1week', 'roll_mean_1h', 'roll_mean_2h', 'roll_mean_1day']

Corrupted columns: ['PULocationID', 'time_bucket', 'trip_count', 'hour', 'minute', 'dayofweek', 'is_weekend', 'month', 'dayofyear', 'weekofyear', 'year', 'slot_of_day', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_holiday', 'cbd_pricing_active', 'borough_id', 'service_zone_id', 'is_airport_zone', 'zone_slot_baseline', 'lag_15min', 'lag_1h', 'lag_2h', 'lag_1day', 'lag_1week', 'roll_mean_1h', 'roll_mean_2h', 'roll_mean_1day']


In [18]:
# Issue #6: Type issues: Integer columns with floats? Strings instead of numbers?
print(f"\nBaseline types:\n{baseline.dtypes}")
print(f"\nCorrupted types:\n{corrupted.dtypes}")

# No issues here. ------------------------------------------


Baseline types:
PULocationID                   int64
time_bucket           datetime64[ns]
trip_count                     int32
hour                           int32
minute                         int32
dayofweek                      int32
is_weekend                      int8
month                          int32
dayofyear                      int32
weekofyear                     int64
year                           int32
slot_of_day                    int32
hour_sin                     float64
hour_cos                     float64
dow_sin                      float64
dow_cos                      float64
month_sin                    float64
month_cos                    float64
is_holiday                      int8
cbd_pricing_active              int8
borough_id                     int64
service_zone_id                int64
is_airport_zone                 int8
zone_slot_baseline           float64
lag_15min                    float32
lag_1h                       float32
lag_2h               